# 04 · Inventory Risk Model

Turns the forecast into decisions: safety stock → reorder point → risk class →
financial impact → prioritised action.

> **Provenance reminder.** Demand is real. Inventory position, unit cost,
> supplier and lead time are *simulated* from documented assumptions
> (`reports/ASSUMPTIONS.md`). Inventory is not random — it is the output of a
> stated replenishment policy run against the real demand series, so historical
> stockouts emerge from genuine demand volatility.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from src.config import load_config, get_paths
cfg, paths = load_config(), get_paths()
print("as-of date:", cfg["project"]["as_of_date"])

as-of date: 2011-12-09


## 1. Safety stock

The textbook form `z · σ_d · √LT` assumes lead time is deterministic. It is not:
supplier lead-time SD ranges 1.0–8.0 days. The combined form is used:

$$SS = z\sqrt{PI \cdot \sigma_d^2 + \bar{d}^2 \cdot \sigma_{LT}^2}$$

where the protection interval `PI = lead_time + review_period` — after a review
you cannot order again until the next one, so exposure runs to the arrival of
the *next* order.

In [2]:
m = pd.read_parquet(paths.processed / "inventory_metrics.parquet")
r = m[m.product_id=="22423"].iloc[0]
z, R = 1.6449, 7
PI = r.lead_time_days + R
hand_ss = z*np.sqrt(PI*r.sd_daily_demand**2 + r.avg_daily_demand**2 * r.lead_time_sd_days**2)
print(f"SKU {r.product_id} — {r.product_name}")
print(f"  lead time {r.lead_time_days}d, sigma_LT {r.lead_time_sd_days}, review {R}d -> PI {PI}d")
print(f"  d_bar {r.avg_daily_demand:.4f} (from forecast), sigma_d {r.sd_daily_demand:.4f}")
print(f"  hand-calculated SS : {hand_ss:8.2f}")
print(f"  module SS          : {r.safety_stock:8.2f}   <- independent verification")
print(f"  reorder point      : {r.reorder_point:8.2f}")

SKU 22423 — REGENCY CAKESTAND 3 TIER
  lead time 24d, sigma_LT 2.32, review 7d -> PI 31d
  d_bar 14.2713 (from forecast), sigma_d 36.4278
  hand-calculated SS :   338.04
  module SS          :   338.00   <- independent verification
  reorder point      :   780.00


### Does modelling lead-time variability actually matter?

In [3]:
from src.inventory_metrics import compute_safety_stock
simple = compute_safety_stock(m.avg_daily_demand, m.sd_daily_demand,
                              m.lead_time_days, m.lead_time_sd_days, 7, z, False)
t = pd.DataFrame({"both_terms": m.safety_stock, "demand_only": simple,
                  "region": m.sourcing_region}).groupby("region").sum(numeric_only=True)
t["uplift_%"] = (100*(t.both_terms/t.demand_only - 1)).round(1)
t.round(0)

,both_terms,demand_only,uplift_%
region,,,
EU Import,110752.0,106777.0,4.0
Far East Import,171386.0,154305.0,11.0
UK Domestic,104059.0,100514.0,4.0


Far-East SKUs need **11% more** safety stock once lead-time variability is modelled, versus 3% for UK domestic. Dropping that term would under-protect exactly the SKUs most exposed to it.

## 2. Risk classification

Two scale-free ratios so a £0.40 sticker and a £12 cakestand are judged alike:

* `cover_ratio  = inventory_position / reorder_point`
* `excess_ratio = inventory_position / expected_requirement_90d`

**Why a 90-day overstock horizon?** This business turns inventory 2.2x a year
(165 days of stock). Judging excess against a 30-day requirement flags 78% of
SKUs — an unusable alert list. The sensitivity of that choice is published rather
than hidden.

In [4]:
pd.read_csv(paths.business_insights / 'overstock_threshold_sensitivity.csv')

,horizon_days,excess_ratio_threshold,n_skus_flagged,pct_skus,excess_value_gbp,pct_of_inventory_value
0,30,1.5,1854,78.2,1547848.0,80.6
1,30,2.0,1328,56.0,1345326.0,70.0
2,30,3.0,899,37.9,1090835.0,56.8
3,60,1.5,1167,49.2,1147778.0,59.8
4,60,2.0,928,39.2,1031140.0,53.7
5,60,3.0,688,29.0,846712.0,44.1
6,90,1.5,950,40.1,978792.0,51.0
7,90,2.0,784,33.1,874358.0,45.5
8,90,3.0,615,25.9,787223.0,41.0


In [5]:
risk = pd.read_parquet(paths.processed / "sku_risk.parquet")
pd.read_csv(paths.risk_tables / "risk_summary.csv")

,risk_level,n_skus,n_actionable,inventory_value,excess_value,estimated_lost_revenue,recommended_order_value,pct_of_skus
0,Critical Stockout Risk,56,51,5938.89,0.00,55998.84,48972.69,2.4
1,Stockout Risk,125,24,24958.67,0.00,6695.30,15272.30,5.3
2,Healthy,1239,0,774672.09,68009.01,0.00,0.00,52.3
3,Overstock Risk,334,285,298594.66,191399.38,0.00,0.00,14.1
4,Severe Overstock,616,579,816775.10,787395.76,0.00,0.00,26.0


## 3. The prioritisation score is not a black box

A weighted sum of percentile-ranked components, weights declared in
`config/config.yaml`. Percentile ranks are used because the raw distributions are
extremely skewed — one SKU with £50k exposure would otherwise dominate.

Every score decomposes back into its parts:

In [6]:
top = risk[risk.risk_family=="Stockout"].nlargest(1,"priority_score").iloc[0]
parts = [c for c in risk.columns if c.startswith("score_part_")]
d = {c.replace("score_part_",""): top[c] for c in parts if pd.notna(top[c]) and top[c]!=0}
print(f"{top.product_name}  (score {top.priority_score:.2f})")
for k,v in d.items(): print(f"  {k:24s} {v:6.2f}")
print(f"  {'SUM':24s} {sum(d.values()):6.2f}")

METAL SIGN TAKE IT OR LEAVE IT   (score 91.52)
  lead_time_pressure         1.63
  revenue_contribution       9.89
  severity                  25.00
  financial_exposure        40.00
  velocity                  15.00
  SUM                       91.52


## 4. Financial impact\n\nBoth figures are **estimates** built on the simulated inventory layer.

In [7]:
print(f"Estimated lost revenue (stockout risk) : GBP {risk.estimated_lost_revenue.sum():>12,.0f}")
print(f"Excess inventory value  (overstock)    : GBP {risk.excess_value.sum():>12,.0f}")
print(f"Dead stock value                       : GBP {risk.loc[risk.is_dead_stock,'inventory_value'].sum():>12,.0f}")
print(f"Total inventory value                  : GBP {risk.inventory_value.sum():>12,.0f}")
print(f"Recommended reorder value              : GBP {risk.order_cost.sum():>12,.0f}")

Estimated lost revenue (stockout risk) : GBP       62,694
Excess inventory value  (overstock)    : GBP    1,046,804
Dead stock value                       : GBP      241,820
Total inventory value                  : GBP    1,920,939
Recommended reorder value              : GBP       64,245


## 5. Recommended actions

In [8]:
recs = pd.read_parquet(paths.processed / "recommendations.parquet")
recs.groupby(["action_type","action_priority"]).agg(
    n_skus=("product_id","size"), value_gbp=("action_value_gbp","sum")
).sort_values("value_gbp", ascending=False).round(0)

,,n_skus,value_gbp
action_type,action_priority,,
Markdown / Clearance,P2 - This month,376,558456.0
Liquidate,P2 - This month,265,241820.0
Reduce Replenishment,P3 - Next cycle,313,186097.0
Reorder Now,P1 - Immediate,34,35801.0
Expedite Reorder,P1 - Immediate,22,20198.0
Reorder,P2 - This week,125,6695.0
No Action,P4 - Monitor,1235,0.0


In [9]:
pd.set_option("display.max_colwidth", 110)
recs[recs.is_actionable].nlargest(5,"action_value_gbp")[
    ["product_name","risk_level","action_type","action_detail","action_value_gbp"]]

,product_name,risk_level,action_type,action_detail,action_value_gbp
18,EDWARDIAN PARASOL PINK,Severe Overstock,Markdown / Clearance,"Run a clearance or promotion on 6,638 excess units (£24,826 of capital). Suspend all replenishment from Pe...",24826.12
43,EDWARDIAN PARASOL RED,Severe Overstock,Markdown / Clearance,"Run a clearance or promotion on 6,461 excess units (£23,712 of capital). Suspend all replenishment from Pe...",23711.87
223,PAPER CHAIN KIT 50'S CHRISTMAS,Severe Overstock,Markdown / Clearance,"Run a clearance or promotion on 13,526 excess units (£22,183 of capital). Suspend all replenishment from S...",22182.64
165,EDWARDIAN PARASOL BLACK,Severe Overstock,Markdown / Clearance,"Run a clearance or promotion on 5,464 excess units (£18,905 of capital). Suspend all replenishment from Pe...",18905.44
24,BLACK/BLUE POLKADOT UMBRELLA,Severe Overstock,Markdown / Clearance,"Run a clearance or promotion on 4,737 excess units (£14,258 of capital). Suspend all replenishment from Rh...",14258.37


## 6. What this system would change

1. **£979k of excess inventory** identified across 950 SKUs — just over half of
   inventory value, concentrated post-Christmas.
2. **265 dead-stock SKUs (£242k)** separated from ordinary overstock, because
   the action is liquidation, not a smaller reorder.
3. **56 SKUs at critical stockout risk**, with per-SKU reorder quantities and
   22 flagged for expediting on long lead times.
4. The incumbent policy's **90.6% fill rate** quantified, with the gap to a 95%
   target attributed to unbuffered demand volatility.